## Cross Validation
- 交差検証によって Logistic Regression と Random Forest を公平に比較

In [1]:
# ライブラリとデータを読み込む

from pathlib import Path
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
from sklearn.model_selection import (
    cross_val_score,
    StratifiedKFold,
    train_test_split
)

In [3]:
# プロジェクトとデータの場所を指定

current_dir = Path.cwd().resolve()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

data_dir = project_root / "data" / "titanic"
output_dir = project_root / "output"

output_dir.mkdir(exist_ok=True)

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

print("train:", train.shape)
print("test:", test.shape)

train: (891, 12)
test: (418, 11)


特徴量エンジニアリング

In [4]:
# "FamilySize"を作成
# Sibsp：一緒に乗っている兄弟・姉妹・配偶者の人数
# Parch：一緒に乗っている親・子供の人数
# 本人も含めるために +1
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

In [5]:
# 一人で乗船しているか
# train["FamilySize"] == 1：1人ならTrue、2人以上ならFalse
# True:1、False：0 に変換
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

In [6]:
# "Name"列から敬称を取り出す
train["Title"] = train["Name"].str.extract(r",\s*([^.]*)\.")
test["Title"] = test["Name"].str.extract(r",\s*([^.]*)\.")

In [7]:
train["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [8]:
# 少数カテゴリをまとめる
common_titles = ["Mr","Miss","Mrs","Master"]

train["Title"] = train["Title"].where(
    train["Title"].isin(common_titles),
    "Rare"
)

test["Title"] = test["Title"].where(
    test["Title"].isin(common_titles),
    "Rare"
)

In [9]:
# 継承によって生存率に差がありそうか
train.groupby("Title")["Survived"].agg(["count","mean"])

,count,mean
Title,,
Master,40,0.575000
Miss,182,0.697802
Mr,517,0.156673
Mrs,125,0.792000
Rare,27,0.444444


In [10]:
# 使用する列を決める
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone",
    "Title"
]
target = "Survived"

X = train[features]
y = train[target]

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone,Title
0,3,male,22.0,1,0,7.2500,S,2,0,Mr
1,1,female,38.0,1,0,71.2833,C,2,0,Mrs
2,3,female,26.0,0,0,7.9250,S,1,1,Miss
3,1,female,35.0,1,0,53.1000,S,2,0,Mrs
4,3,male,35.0,0,0,8.0500,S,1,1,Mr


In [11]:
# 数値列とカテゴリ列を分けて、それぞれ前処理
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title"
]

In [12]:
# 数値列
# 欠損値を中央値で補う
numeric_transformer = Pipeline(
    steps=[
        # SimpleImputer(strategy="median") は、fit時に学習データから各列の中央値を計算し、
        # その後のtransform時にその値を使って欠損値を補う
        # X_valid（検証データ）は、本来、学習時には知らないはずであるため、中央値計算などで用いない
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

In [13]:
# カテゴリ列
# 欠損値を最頻値で補い、文字列を数値に変換
categorical_transformer = Pipeline(
    steps=[
        # strategy="most_frequent" によって、最頻値で欠損値処理を行う
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            # One-Hot Encodingでは、１つのカテゴリ列を、カテゴリごとの０・１の列へ変換する
            "onehot",
            OneHotEncoder(
                # 学習データにはなかったカテゴリーが検証データや将来のデータに表れた場合に、
                # handle_unknown="ignore" があることで、エラーにせず、
                # その変数に対応する One-Hot列をすべて０として扱う
                handle_unknown="ignore",
                # One-Hot Encoding後のデータを、通常の密な配列として出力する
                sparse_output=False,
            ),
        ),
    ]
)

In [14]:
# 前処理をまとめる
# ColumnTransformer：列の種類ごとに、別々の前処理を適用する
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [ ]:
# 前処理とモデルを一体化する
# Pipelineを用いることで、前処理 → Random_Forest としている
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", 
         RandomForestClassifier(
             # 決定木を200本作る
             n_estimators=200,
             random_state=42
             )
        ),
    ]
)

In [16]:
# 5分割するルールを作成する

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [17]:
scores_model = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

In [18]:
print(scores_model)

[0.81005587 0.78651685 0.80337079 0.82022472 0.83146067]


過学習を起こしている?

In [19]:
# 前処理とモデルを一体化する
# Pipelineを用いることで、前処理 → ロジスティック回帰 としている
# 細かく分かれすぎるのを防ぐversion
model_over = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", 
         RandomForestClassifier(
             # 決定木を200本作る
             n_estimators=200,
             # 1本の決定木を深くしすぎない
             max_depth=5,
             # 木の末端のグループには最低３人必要
             min_samples_leaf=3,
             random_state=42
             )
        ),
    ]
)

In [31]:
scores_model_over = cross_val_score(
    model_over,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

In [32]:
print(scores_model_over)

[0.83798883 0.82022472 0.83146067 0.83146067 0.83707865]


線形回帰はどうなのか？

In [25]:
from sklearn.linear_model import LogisticRegression

In [26]:
# 前処理とモデルを一体化する
# Pipelineを用いることで、前処理 → ロジスティック回帰 としている
model_log = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

In [28]:
scores_model_log = cross_val_score(
    model_log,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

In [29]:
print(scores_model_log)

[0.83798883 0.81460674 0.8258427  0.83146067 0.83146067]


In [33]:
# それぞれの平均・標準偏差を確認
print(
    "model：",
    round(scores_model.mean(),4),
    "+/-",
    round(scores_model.std(),4)
)

print(
    "model_over：",
    round(scores_model_over.mean(),4),
    "+/-",
    round(scores_model_over.std(),4)
)

print(
    "model_log：",
    round(scores_model_log.mean(),4),
    "+/-",
    round(scores_model_log.std(),4)
)

model： 0.8103 +/- 0.0152
model_over： 0.8316 +/- 0.0063
model_log： 0.8283 +/- 0.0078


model_overが最も精度が高いと結論付けられた